# CoordinateSystem — parameter combinations

Runnable examples showing how the main `CoordinateSystem` parameters change the
result. Each view is rendered serverlessly with `display_snapshot()`, so the 3D
views can still be rotated/zoomed inline.

For the full reference, see `coordinate-system.md`.


In [ ]:
import math

from IPython.display import display

from pytanga.geometry import Point
from pytanga.viz import (
    CameraConfig3d,
    CoordinateSystem,
    PointPath,
    PointPathStyle,
    PointStyle,
    Visualizer,
)


def snapshot(title, space_dim, cs_kwargs, plot=None, marker=None, camera=None, height=340):
    viz = Visualizer(
        title=title,
        space_dim=space_dim,
        add_default_axes=False,
        add_default_grid=False,
        camera=camera,
    )
    cs = CoordinateSystem(viz, **cs_kwargs)
    if marker is not None:
        viz.new(Point(*marker), color="#ffffff", style=PointStyle(size=0.15), label="anchor")
    if plot is not None:
        xs, ys, kwargs = plot
        cs.plot(xs, ys, **kwargs)
    return viz.display_snapshot(height=height)


## Scales: linear vs log

`xscale`/`yscale` switch between linear and logarithmic tick spacing (any base).
The world stays linear; only labels and grid spacing change.


In [ ]:
xs = [0.05 * i for i in range(101)]
display(snapshot("Linear", 2, dict(xlim=(-0.5, 5), ylim=(-1.2, 1.2)),
                 plot=(xs, [math.sin(x) for x in xs], {"color": "#ffcc00"})))

lx = [0.1 * (10 ** (0.1 * i)) for i in range(30)]
display(snapshot("Log-log (base 10)", 2,
                 dict(xlim=(0.1, 100), ylim=(0.1, 100), xscale="log", yscale="log"),
                 plot=(lx, [x * x for x in lx], {"color": "#ffcc00"})))


## External size vs data range

`size` fixes the physical extent of the plane; the data range is stretched onto
it. Here the range `0..4π` is squeezed into a 2-unit-wide plane.


In [ ]:
xs = [0.05 * i for i in range(252)]
display(snapshot("size=(2, 1) shows 0..4π", 3,
                 dict(xlim=(0, 4 * math.pi), ylim=(-1.5, 1.5), size=(2.0, 1.0),
                      position=(0, 0, 0), normal=(0, 0, 1), up=(0, 1, 0)),
                 plot=(xs, [math.sin(x) for x in xs], {"color": "#ffcc00"})))


## Align

`align` decides which point of the plane sits at `position` (the white anchor
point): `(0, 0)` = bottom-left corner, `(1, 1)` = top-right.


In [ ]:
camera = CameraConfig3d(position=(2.2, 1.8, 2.6), target=(0.4, 0.3, 0))
xs = [0.05 * i for i in range(81)]
for align in [(0, 0), (0.5, 0.5), (1, 1)]:
    display(snapshot(
        f"align={align}",
        3,
        dict(xlim=(0, 4), ylim=(-2.2, 2.2), size=(2.0, 1.0),
             align=align, position=(0, 0, 0), normal=(0, 0, 1), up=(0, 1, 0)),
        plot=(xs, [2 * math.sin(x) for x in xs], {"color": "#ffcc00"}),
        marker=(0, 0, 0),
        camera=camera,
    ))


## Axis crossing (`axis_origin`)

By default the axes form a spine (bottom + left). `axis_origin` moves the point
where the two axes cross (in data coordinates).


In [ ]:
xs = [0.05 * i for i in range(101)]
display(snapshot("Default spine", 2, dict(xlim=(-0.5, 5), ylim=(-1.2, 1.2)),
                 plot=(xs, [math.sin(x) for x in xs], {"color": "#ffcc00"})))
display(snapshot("axis_origin=(0, 0)", 2,
                 dict(xlim=(-0.5, 5), ylim=(-1.2, 1.2), axis_origin=(0, 0)),
                 plot=(xs, [math.sin(x) for x in xs], {"color": "#ffcc00"})))


## Live trail with auto-scaling time axis

A `PointPath` registered with `add_plot(..., auto_x=True)` maps data onto the
plane and fits the x axis to the trail's time range (`min_x_span=5` keeps at
least a few seconds visible). In a live loop you call `cs.update_plots()` +
`viz.flush()` each frame.


In [ ]:
trail = PointPath(max_points=400)
for i in range(200):
    t = i * 0.05
    trail.add((t, 1.5 * math.exp(-0.2 * t) * math.sin(3.0 * t)))

viz = Visualizer(title="Trail with auto time axis", space_dim=3,
                 add_default_axes=False, add_default_grid=False)
cs = CoordinateSystem(viz, xlim=(0, 10), ylim=(-2, 2), size=(2.5, 1.2),
                      labels=("time (s)", "value"), position=(0, 0, 0),
                      normal=(0, 0, 1), up=(0, 1, 0))
cs.add_plot(trail, color="#ffcc00", style=PointPathStyle(line_thickness=2), auto_x=True)
cs.update_plots()
viz.display_snapshot(height=340)
